[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/lin-elastic_mixed-BC.ipynb)

# Two-Phase Composite RVE — Mixed Strain/Stress Boundary Conditions

A more realistic walkthrough of FFTjax's mechanical solver: instead of prescribing the full
macroscopic strain tensor (as in [`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb)), this
notebook prescribes a **mixed** boundary condition — displacement-controlled tension along x with
free (traction-free) lateral surfaces along y and z — the condition an actual tensile-test
specimen is under (axial extension imposed at the grips, lateral surfaces free to contract via
Poisson's effect) — rather than the artificially-constrained uniaxial *strain* condition used
there, which locks the lateral strains to zero. The point of this setup is to actually measure the
composite's effective Poisson's ratios, which the fully-constrained case cannot do.

We want to solve the mechanical equilibrium problem on this composite subject to its governing
PDE constraints:

$$
\nabla \cdot \sigma(\mathbf{x}) = 0, \qquad
\sigma = \mathbb{C}(\mathbf{x}):\varepsilon, \qquad
\varepsilon = \tfrac{1}{2}\big(\nabla u + \nabla u^\top\big)
$$

on a periodic domain. Instead of prescribing the full macroscopic strain
$\bar\varepsilon = \langle\varepsilon(\mathbf{x})\rangle_\Omega$, each tensor component $ij$ is
independently either strain- or stress-controlled, selected by a mask $c_{ij}\in\{0,1\}$ (`control`
in the code):

$$
\langle\varepsilon_{ij}(\mathbf{x})\rangle_\Omega = \bar\varepsilon_{ij} \quad (c_{ij}=0), \qquad
\langle\sigma_{ij}(\mathbf{x})\rangle_\Omega = \bar\sigma_{ij} \quad (c_{ij}=1)
$$

For this notebook: $c_{11}=0$ (axial strain prescribed, $\bar\varepsilon_{11}=10^{-3}$), $c_{22}=c_{33}=1$
(lateral surfaces free, $\bar\sigma_{22}=\bar\sigma_{33}=0$), all shear components strain-controlled
at zero -- exactly the `eps_bar`/`control`/`stress_goal` triple passed to `solve_mechanics` below.

The solver is displacement-based: the unknown is the periodic displacement fluctuation
$\hat{u}$ itself, with the strain built directly from its symmetric gradient in Fourier space and
the true (possibly heterogeneous) stiffness $\mathbb{C}(\mathbf{x})$ applied directly -- no
reference-medium approximation, unlike the Lippmann-Schwinger scheme in
[`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb). That's exactly what makes
stress-controlled macroscopic directions ($c_{ij}=1$) possible at all: the free-surface correction
is embedded as an extra unknown in the same CG system, which the reference-medium scheme cannot do.

This uses `problems.mechanics.solve_mechanics(..., formulation="displacement", control=...)` --
required here because the reference-medium (`lippmann_schwinger`) formulation can't do
stress-controlled macroscopic directions at all (see that function's docstring).

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git@refactor/target-layout
else:
    import sys
    sys.path.insert(0, "../src")
    print("Running locally — using the local src/ checkout.")

import os
import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Generate the composite RVE

`generation.rve.make_square_composite_rve` builds a square-packed 2-fiber RVE (GFRP): a matrix phase with
circular fiber cross-sections arranged on a square lattice, extruded along Z into a 3-D voxel grid. Here we use a 1-voxel-thick RVE, with a fiber volume fraction of 0.5, fiber radius of 5 μm, and voxel spacing of 0.2 μm.

Setting `nz=1` reduces the geometry to a 2-D-like slab (uniformly extruded along Z); combined with a macroscopic strain that has no out-of-plane (Z) components -- as prescribed below -- this gives a plane-strain solve.

In [ ]:
from generation.rve import make_square_composite_rve

phi     = 0.5      # target fiber volume fraction
r_fiber = 0.005     # fiber radius [mm]
dx      = 0.0002    # target voxel size [mm]

phase_np, n, L, phi_act = make_square_composite_rve(
    phi=phi,
    r_fiber=r_fiber,
    dx=dx,
    N_min=32,       # minimum number of voxels in x, y direction 
    nz=1,           # number of voxels in z direction (thickness) / along fiber axis
)


print("grid n :", n)
print("total voxels Nv:", int(np.prod(n)))
print("domain L [mm]:", tuple(f"{float(Li):.5g}" for Li in L))
print("fiber volume fraction (actual):", f"{phi_act:.4f}")

In [ ]:
# Visualize the fiber cross-section in the XY plane (Z=0)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r")
ax.set_title(f"Fiber cross-section (Vf={phi_act:.3f})")
ax.set_xlabel("x [voxel]")
ax.set_ylabel("y [voxel]")
plt.show()

## Materials

We define two isotropic materials — a glass fiber and an epoxy matrix — a common, high-contrast
(~23x stiffness ratio) composite.

Every material model in `materialmodels/` implements a common `ConstitutiveModel` interface: a
`.stiffness_tensor()` method that returns its 4th-order stiffness tensor $\mathbb{C}$ (so that
$\sigma = \mathbb{C}:\varepsilon$), regardless of the underlying parametrization —
`LinearElasticIsotropic` here (from $E$, $\nu$), a transversely isotropic model elsewhere. That
common return type is what lets `assemble_C_field` gather a per-voxel $\mathbb{C}(\mathbf{x})$
field from any mix of material models via the phase index, without needing to know which
parametrization each one uses.

In [ ]:
from materialmodels.elastic.isotropic import LinearElasticIsotropic
from materialmodels.assembly import describe_materials

matrix = LinearElasticIsotropic(E=3.0e3,  nu=0.35, name="epoxy matrix")
fiber  = LinearElasticIsotropic(E=70.0e3, nu=0.20, name="glass fiber")
materials = [matrix, fiber]   # index 0 = matrix, 1 = fibre -- matches the phase labels below

phase = jnp.array(phase_np.reshape(-1))   # (nx,ny,nz) -> (Nv,), see markdown above

describe_materials(materials)

## Mixed boundary conditions and solve

Prescribe displacement-controlled tension along x and traction-free lateral surfaces along y, z --
the standard setup for measuring an effective Poisson's ratio in a virtual tensile test:

- `control[0][0] = 0` (strain-controlled): `eps_bar[0, 0] = 1e-3` sets the axial tension directly.
- `control[1][1] = control[2][2] = 1` (stress-controlled): `stress_goal` is zero there, i.e. free
  lateral surfaces -- the solver finds whatever lateral strain makes `sigma22 = sigma33 = 0`.
- Shear components stay strain-controlled at zero (no shear loading).

`solve_mechanics(..., formulation="displacement", control=control)` solves for the displacement
fluctuation and the free macroscopic strain components jointly, with no reference-medium
approximation -- the true heterogeneous `C_field` (assembled internally from `materials`/`phase`,
same as `lin-elastic_strain.ipynb`) is used directly. It returns `list[IncrementResult]` -- one
element here, at `t=1.0` -- so `results[0].solution` is the `ElasticitySolution`, whose
`.eps`/`.sigma`/`.eps_bar`/`.converged` attributes we read below; unlike the pure-strain case,
`.eps_bar` here is the *solved* macroscopic strain (the lateral contraction), not `None`.

For the full mathematical derivation of the mixed-BC formulation -- how the stress-controlled
correction is embedded as an extra unknown in the CG system -- see
[Mechanical Solvers](https://choROPeNt.github.io/FFTjax/documentation/theorie/mechanical) in the
documentation. The two Poisson's ratios printed below follow directly from the solved lateral
strains.

In [ ]:
from problems.mechanics import solve_mechanics

eps_bar = jnp.array([
    [1.0e-3, 0.0, 0.0],
    [0.0,  0.0, 0.0],
    [0.0,  0.0, 0.0],
]) # xx and shear entries (control == 0, strain-controlled) are used
control = (
    (0, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
)  # xx strain-controlled (tension); yy, zz stress-controlled (free lateral surfaces); shear strain-controlled = 0
stress_goal = jnp.array([
    [0.0, 0.0, 0.0],
    [0.0,  0.0, 0.0],
    [0.0,  0.0, 0.0],
]) # only yy, zz entries (control == 1) are used -- both 0, i.e. free lateral surfaces

results = solve_mechanics(
    n, L, phase, materials, eps_bar,
    formulation="displacement", control=control, stress_goal=stress_goal,
    toler_lin=1e-6, maxiter=100,
)
sol = results[0].solution
eps, sigma, converged, eps_bar_out = sol.eps, sol.sigma, sol.converged, sol.eps_bar

print("converged     :", bool(converged))
print("eps_bar (solved):")
print(np.array(eps_bar_out))
print()
print("sigmaxx (avg) :", float(jnp.mean(sigma[0, 0])), "MPa")
print("sigmayy (avg) :", float(jnp.mean(sigma[1, 1])), "MPa  (target: 0, free surface)")
print("sigmazz (avg) :", float(jnp.mean(sigma[2, 2])), "MPa  (target: 0, free surface)")
print()
print("effective nu_xy = -epsyy/epsxx :", float(-eps_bar_out[1, 1] / eps_bar_out[0, 0]))
print("effective nu_xz = -epszz/epsxx :", float(-eps_bar_out[2, 2] / eps_bar_out[0, 0]))

The free lateral surfaces let the composite contract under axial load — exactly what a real
tensile specimen does (Poisson's effect) — which the constrained uniaxial-*strain* case in
`lin-elastic_strain.ipynb` cannot show, since it locks `eps22 = eps33 = 0` by construction. The two
effective Poisson ratios above also differ from each other: the square-packed fibres (aligned along
Z) make this composite's in-plane (xy) and through-thickness (xz) responses genuinely anisotropic,
not the single-value isotropic Poisson's ratio either constituent has on its own.

## Post-processing

Now we can visualize the results and also export them as a `.xdmf`/`.h5` pair for further
post-processing in ParaView or other visualization software, via FFTjax's `IncrementalWriter`
(the project-wide standard for field-data output). Every field here -- displacement, strain,
stress, phase -- is evaluated on the same voxel grid, so all of them are written voxel-centered
(`Center="Cell"`); there's no FEM-style node/cell split in this spectral scheme, so there's nothing
to gain from writing displacement at a different resolution than everything else.

In [ ]:
from post.fields import field_to_grid, von_mises, compute_displacement, to_voigt

eps_grid   = field_to_grid(eps, n)
sigma_grid = field_to_grid(sigma, n)
u_grid     = compute_displacement(eps, eps_bar_out, n, L)
vm_grid    = von_mises(sigma_grid)

eps_voigt   = to_voigt(eps_grid).astype(np.float64)
sigma_voigt = to_voigt(sigma_grid).astype(np.float64)

In [ ]:
from utils.plotting import FieldPanel, plot_field_grid

VOIGT_LABELS = ["x", "y", "z", "xy", "xz", "yz"]
extent = [0.0, L[0], 0.0, L[1]]  # physical [mm] extent

phase_panel = FieldPanel(phase_np[:, :, 0], "Fiber phase", cmap="gray_r")
disp_panels = [
    FieldPanel(u_grid[:, :, 0, 0], r"Displacement $u_x$ [mm]", fmt="%.1e"),
    FieldPanel(u_grid[:, :, 0, 1], r"Displacement $u_y$ [mm]", fmt="%.1e"),
]

strain_row, stress_row = [], []
for i in [0, 1, 3]:  # normal-x, normal-y, shear-xy Voigt components
    is_shear = i == 3  # shear component: report engineering shear strain gamma = 2*epsilon
    strain_label = rf"$\gamma_{{{VOIGT_LABELS[i]}}}$" if is_shear else rf"$\varepsilon_{{{VOIGT_LABELS[i]}}}$"
    stress_label = rf"$\tau_{{{VOIGT_LABELS[i]}}}$" if is_shear else rf"$\sigma_{{{VOIGT_LABELS[i]}}}$"
    strain_row.append(FieldPanel(eps_voigt[:, :, 0, i], f"Strain {strain_label}", fmt="%.1e"))
    stress_row.append(FieldPanel(sigma_voigt[:, :, 0, i], f"Stress {stress_label}", fmt="%.1f"))

fig, axes = plot_field_grid([[phase_panel, *disp_panels], strain_row, stress_row], extent, figsize=(10, 9))
plt.show()

In [ ]:
from utils.io.xdmf_writer import IncrementalWriter

output_dir = "output" if IN_COLAB else "../output/notebooks"  

os.makedirs(output_dir, exist_ok=True)

with IncrementalWriter(f"{output_dir}/composite_rve_mixed_bc", grid_shape=n, grid_length=L) as w:
    w.write_increment(0, {
        "phase":        phase_np.astype(np.float64),
        "displacement": u_grid.astype(np.float64),
        "strain":       to_voigt(eps_grid).astype(np.float64),
        "stress":       to_voigt(sigma_grid).astype(np.float64),
        "von_mises":    vm_grid.astype(np.float64),
    }, time=0.0)

print(f"Wrote {output_dir}/composite_rve_mixed_bc.h5")
print(f"      {output_dir}/composite_rve_mixed_bc.xdmf")
print("Open the .xdmf in ParaView with the 'Xdmf3ReaderT' reader.")

## Next steps

- Compare against [`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb)'s pure uniaxial-*strain*
  case on the same geometry/materials — same load direction, very different lateral response.
- Use ['lin-elastic_strain_vmap'](./lin-elastic_strain_vmap.ipynb) for parallelized solves on a bathc of different macroscopic strain/stress conditions (e.g. a full load curve) on the same geometry/materials.
- See `problems.fracture.solve_fracture` for coupling this kind of mechanical solve
  (`formulation="displacement"` is supported there too, via the same `control` argument) to
  phase-field damage evolution.